# Decompose → Per-Sub-Issue Funnel → Reranker → F1 (val_004 PoC)

**Goal**: prove that decomposing a multi-aspect val query into atomic single-issue sub-queries lifts Macro F1 from ~0.05 (Stage B v3 baseline) toward 0.4+.

**Why this should work** — the fine-tuned Qwen3-Reranker was trained on ~100-300 char single-issue queries (train.csv distribution). val.csv queries are 1000-1700 chars and aggregate 3-16 distinct legal issues. Decomposing snaps the reranker input back inside its training distribution.

**Why val_004** — smallest val query (10 gold cits, est. 3 sub-issues), no val_009 LoRA leakage, cheapest end-to-end PoC.

**No hardcoding**: every K, every issue, every target is LLM-derived from this query alone. Architecture generalizes to any val/test/production query.

**Self-contained**: the v7.4/v7.5 14-channel funnel module is inlined in Phase 3b. Upload only this `.ipynb` — no separate `.py` files needed.

**Phases** (all disk-checkpointed; safe to interrupt and resume):
1. Env + load val_004
2. vLLM Qwen3-32B-AWQ → decompose / translate / targets
3. 14-channel v7.4/v7.5 funnel per (sub-issue × {EN, DE}) → top-200 per sub-issue
4. Qwen3-Reranker-8B + val_009 LoRA → top-K per sub-issue
5. Union → Kaggle exact-string F1

**Channels in this funnel** (14, matches v7.4 notebook):
law_direct_match, court_statute, co_citation, per_area_bedrock, statute_backprop,
sibling_expansion, graph_forward, graph_reverse, graph_2hop (disabled by config),
concept_en, term_orig, bm25 (multi-lang FTS5), vector_raw, vector_enriched.

v7.5's 15th channel (concept_cosine) and 16th channel (HyDE-aspect multi-query) are
**downstream** of this funnel — applied on top of these results when building Stage A v3
features. For the PoC we omit them; if F1 is close-but-short we can add them as a post-funnel pass.


## Phase 0 — Environment sanity check

Strict env per the reranker-finetune-poc lessons. Stale-kernel detection up top so we fail fast on the wrong transformers version.

In [ ]:
# Phase 0a — detect stale kernel from a previous notebook in same Colab session
import sys, importlib, os, subprocess

REQUIRED_TRANSFORMERS = "4.56.2"
already_loaded = "transformers" in sys.modules
if already_loaded:
    import transformers
    if transformers.__version__ != REQUIRED_TRANSFORMERS:
        raise RuntimeError(
            f"Stale kernel: transformers {transformers.__version__} already imported, "
            f"need {REQUIRED_TRANSFORMERS}. RESTART RUNTIME and run Phase 0 first."
        )

# Phase 0b — uninstall conflict packages BEFORE installing pins
# These break either peft (torchao), sentence_transformers (torchcodec), or
# bitsandbytes (cu13 lib) per memory of the reranker finetune debug session.
def pip(cmd):
    print(f"+ pip {cmd}")
    subprocess.check_call([sys.executable, "-m", "pip"] + cmd.split())

for pkg in ["torchao", "torchcodec", "bitsandbytes"]:
    try:
        pip(f"uninstall -y {pkg}")
    except Exception:
        pass

# Phase 0c — install strict pins
pip(f"install -q transformers=={REQUIRED_TRANSFORMERS}")
pip("install -q peft==0.13.2 vllm==0.6.3.post1 accelerate==1.0.1")
pip("install -q sentence-transformers==3.2.1")  # for vector_raw / vector_enriched channels
pip("install -q pandas==2.2.3 pyarrow==17.0.0")
print("Phase 0 complete. Restart NOT needed if no errors above.")


In [ ]:
# Phase 0d — env truth report (after install)
import importlib, torch
for mod in ["transformers", "peft", "vllm", "accelerate", "torch"]:
    try:
        m = importlib.import_module(mod)
        print(f"  {mod:15s} {getattr(m, '__version__', '?')}")
    except ImportError as e:
        print(f"  {mod:15s} MISSING ({e})")
print(f"  cuda            available={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"  PYTORCH_CUDA_ALLOC_CONF={os.environ.get('PYTORCH_CUDA_ALLOC_CONF', '<unset>')}")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("  set PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True")


## Phase 1 — Mount, paths, load val_004

**This notebook is self-contained.** The 14-channel funnel module is inlined in Phase 3b — you don't upload any `.py` files alongside this notebook.

**Drive layout expected** (auto-detected via `first_existing` — works with either local-mirror or Drive layout):

```
/content/drive/MyDrive/swiss_law/
  data/val.csv
  data/checkpoints/law_llm_descriptors_0000000_all.jsonl    (Drive layout)
    -- OR --
  llm_enrichment_output_law_173k/law_llm_descriptors_0000000_all.jsonl  (local-mirror)

  artifacts_v2/court_authority_cards_v5_unified.jsonl       (Drive)
    -- OR -- artifacts/court_authority_cards_v5_unified.jsonl

  artifacts/embeddings/qwen3_8b_unified_chunk{000..026}.npy (Drive)
    -- OR -- embeddings/qwen3_8b_unified_chunk*.npy

  artifacts/embeddings/qwen3_8b_unified_manifest.parquet
    -- OR -- embeddings/qwen3_8b_unified_manifest.parquet

  data_insights/citation_graph_extracted.sqlite             (Drive flat)
    -- OR -- data_insights/citation_graph_db_and_edges/citation_graph_extracted.sqlite

  research/
    decompose_rerank_poc/                          (output dir, created)
    reranker_finetune_poc_val009/lora_adapter/     (LoRA weights)
```

Phase 1's path probe prints OK / MISSING for every required artifact so you know up front whether Phase 3 will succeed.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
DRIVE_ROOT      = Path("/content/drive/MyDrive/swiss_law")
DATA_DIR        = DRIVE_ROOT / "data"
OUT_DIR         = DRIVE_ROOT / "research" / "decompose_rerank_poc"
OUT_DIR.mkdir(parents=True, exist_ok=True)
LORA_DIR        = DRIVE_ROOT / "research" / "reranker_finetune_poc_val009" / "lora_adapter"
FUNNEL_ART_ROOT = DRIVE_ROOT     # data/, artifacts/, etc. are siblings under here

# This notebook is for val_004 only — single-query PoC
QID = "val_004"

# Checkpoint files (each phase writes one, the next reads it)
CKPT_BUNDLE     = OUT_DIR / f"{QID}_bundle.json"      # Phase 2 output (sub-issues + targets)
CKPT_CANDIDATES = OUT_DIR / f"{QID}_candidates.parquet"  # Phase 3 output (top-200 per sub-issue)
CKPT_RERANKED   = OUT_DIR / f"{QID}_reranked.parquet"  # Phase 4 output (top-K per sub-issue)
CKPT_F1         = OUT_DIR / f"{QID}_f1.json"           # Phase 5 output

print(f"OUT_DIR:         {OUT_DIR}")
print(f"LORA_DIR:        {LORA_DIR} {'(OK)' if LORA_DIR.exists() else '(MISSING)'}")

# Probe each artifact at BOTH known layouts (local-mirror vs Drive). The funnel
# module also resolves via first_existing(), so we just need any one of these to exist per row.
PROBE = [
    ("val.csv",                ["data/val.csv"]),
    ("law LLM descriptors",    ["data/checkpoints/law_llm_descriptors_0000000_all.jsonl",
                                "llm_enrichment_output_law_173k/law_llm_descriptors_0000000_all.jsonl"]),
    ("court cards v5",         ["artifacts_v2/court_authority_cards_v5_unified.jsonl",
                                "artifacts/court_authority_cards_v5_unified.jsonl"]),
    ("embeddings manifest",    ["artifacts/embeddings/qwen3_8b_unified_manifest.parquet",
                                "embeddings/qwen3_8b_unified_manifest.parquet"]),
    ("embeddings chunk dir",   ["artifacts/embeddings", "embeddings"]),
    ("citation graph sqlite",  ["data_insights/citation_graph_extracted.sqlite",
                                "data_insights/citation_graph_db_and_edges/citation_graph_extracted.sqlite",
                                "artifacts/citation_graph_extracted.sqlite"]),
]
missing_artifacts = []
for name, candidates in PROBE:
    found = None
    for sub in candidates:
        p = FUNNEL_ART_ROOT / sub
        if p.exists():
            found = p
            break
    if found is None:
        print(f"  [MISSING] {name:25s}  tried: {candidates}")
        missing_artifacts.append(name)
    else:
        print(f"  [OK]      {name:25s}  {found.relative_to(FUNNEL_ART_ROOT)}")

if missing_artifacts:
    print(f"\nBLOCKING: {len(missing_artifacts)} artifact(s) not found anywhere — funnel will fail in Phase 3.")
    print("Fix by either: (a) uploading the missing files to one of the listed candidate paths, or")
    print("(b) telling me the actual Drive path so I can add it to the candidate list.")
else:
    print("\nAll funnel artifacts located. Ready for Phase 3.")


In [ ]:
import pandas as pd

val_df = pd.read_csv(DATA_DIR / "val.csv")
row = val_df[val_df.query_id == QID].iloc[0]
PARENT_QUERY = row["query"]
PARENT_GOLD  = [c.strip() for c in row["gold_citations"].split(";") if c.strip()]

print(f"Query ID:      {QID}")
print(f"Query length:  {len(PARENT_QUERY)} chars")
print(f"Gold count:    {len(PARENT_GOLD)}")
print(f"\nQuery preview:\n{PARENT_QUERY[:400]}...")
print(f"\nFirst 5 gold:\n  " + "\n  ".join(PARENT_GOLD[:5]))


## Phase 2 — Decompose / translate / targets (vLLM)

One vLLM session, three sequential prompts:
1. **Decompose**: parent query → list of atomic legal issues in English
2. **Translate**: each issue → German legal-style sub-query
3. **Targets**: each issue → `{statutes, concepts_en, term_orig, legal_areas}` for funnel

Output: `{QID}_bundle.json` with everything needed for Phase 3.

**Important**: kill the vLLM engine after this phase to free GPU for the funnel + reranker.

In [ ]:
# Skip Phase 2 if bundle already exists
import json
if CKPT_BUNDLE.exists():
    print(f"[skip] {CKPT_BUNDLE} already exists. Delete to re-run Phase 2.")
    bundle = json.loads(CKPT_BUNDLE.read_text(encoding="utf-8"))
    print(f"  loaded {len(bundle['sub_issues'])} sub-issues from cache")
else:
    bundle = None


In [ ]:
# Phase 2a — spin up vLLM (skip if bundle cached)
if bundle is None:
    from vllm import LLM, SamplingParams

    LLM_MODEL = "Qwen/Qwen3-32B-AWQ"
    llm = LLM(
        model=LLM_MODEL,
        quantization="awq",
        max_model_len=8192,
        gpu_memory_utilization=0.85,
        enforce_eager=False,
        dtype="auto",
    )
    print(f"vLLM loaded: {LLM_MODEL}")


In [ ]:
# Phase 2b — decomposition prompt
DECOMPOSE_PROMPT = """You are a Swiss legal expert. The user query below combines several distinct legal issues. List each atomic legal issue as a separate single-sentence English question.

Rules:
- Each issue must be SINGLE-ASPECT — answerable by 1-8 statute articles and/or BGE decisions.
- Each issue must be self-contained — readable WITHOUT the parent query.
- Use Swiss legal terminology where natural (e.g., "pre-trial detention", "principle of proportionality").
- Output a JSON array of strings. Nothing else. No explanations.

Parent query:
{query}

JSON array of atomic issues:"""

def chat_complete(prompt, max_tokens=2048, temperature=0.0, json_schema=None):
    """Single-turn chat with Qwen3-32B-AWQ via vLLM's chat() API.

    Uses the model's chat template (wraps the message as
    <|im_start|>user ... <|im_end|><|im_start|>assistant so vLLM has a real
    stop token), and disables Qwen3's hybrid thinking mode so we don't pay
    for <think>...</think> blocks on extraction tasks.

    If json_schema is provided, uses vLLM's guided decoding to force the
    output to conform to that JSON Schema — eliminates regex parsing hacks.
    """
    sp = SamplingParams(temperature=temperature, max_tokens=max_tokens)
    if json_schema is not None:
        try:
            from vllm.sampling_params import GuidedDecodingParams
            sp.guided_decoding = GuidedDecodingParams(json=json_schema)
        except ImportError:
            pass  # older vLLM — fall through
    messages = [{"role": "user", "content": prompt}]
    out = llm.chat(
        messages,
        sp,
        chat_template_kwargs={"enable_thinking": False},
    )
    return out[0].outputs[0].text

if bundle is None:
    # Force a JSON array of strings — eliminates the greedy-regex problem entirely
    raw = chat_complete(
        DECOMPOSE_PROMPT.format(query=PARENT_QUERY),
        json_schema={"type": "array", "items": {"type": "string"}},
    )
    print("Raw decomposition output:")
    print(raw[:2000])

    # With guided_decoding, raw is already valid JSON. Keep a non-greedy regex
    # fallback for older vLLM versions that don't support guided decoding.
    import re
    try:
        sub_issues_en = json.loads(raw.strip())
    except json.JSONDecodeError:
        m = re.search(r"\[.*?\]", raw, re.DOTALL)
        if not m:
            raise RuntimeError(f"No JSON array in decomposition output:\n{raw}")
        sub_issues_en = json.loads(m.group(0))

    if not (isinstance(sub_issues_en, list) and all(isinstance(x, str) for x in sub_issues_en)):
        raise RuntimeError(f"Expected list[str], got: {sub_issues_en!r}")

    print(f"\n>>> Decomposed into {len(sub_issues_en)} sub-issues")
    for i, s in enumerate(sub_issues_en):
        print(f"  [{i}] {s}")


In [ ]:
# Phase 2c — translate each sub-issue to German
TRANSLATE_PROMPT = """Translate this English Swiss legal question into German, using the formal German legal style of Swiss federal court judgments. Preserve all statute references and legal terms exactly. Output ONLY the German translation, nothing else.

English: {issue}

German:"""

if bundle is None:
    sub_issues_de = []
    for issue in sub_issues_en:
        de = chat_complete(TRANSLATE_PROMPT.format(issue=issue), max_tokens=512).strip()
        # strip any "German:" prefix and surrounding quotes
        de = de.split("\n")[0].strip().strip('"').strip("'")
        sub_issues_de.append(de)
    print(f">>> Translated {len(sub_issues_de)} sub-issues")
    for en, de in zip(sub_issues_en, sub_issues_de):
        print(f"  EN: {en}")
        print(f"  DE: {de}\n")


In [ ]:
# Phase 2d — target generation per sub-issue
TARGETS_PROMPT = """For this Swiss legal question, identify the SPECIFIC retrieval targets that would be needed to answer it from the Swiss federal law corpus.

Question: {issue}

Output a JSON object with these keys (omit any that are not applicable; do not invent things):
- "statutes": list of canonical statute references like ["Art. 221 StPO", "Art. 117 StGB"] — ONLY if you are confident specific articles apply
- "concepts_en": list of English legal concept terms like ["pre-trial detention", "collusion risk"]
- "term_orig": list of German legal terms like ["Untersuchungshaft", "Kollusionsgefahr"]
- "legal_areas": list of area codes from {{StPO, StGB, ZGB, OR, BV, SchKG, BGG, ZPO, IPRG, UWG, MWSTG, DBG, AHV, KVG, USG, RPG, MietR, BankG}}

JSON only, no commentary:"""

TARGETS_SCHEMA = {
    "type": "object",
    "properties": {
        "statutes":     {"type": "array", "items": {"type": "string"}},
        "concepts_en":  {"type": "array", "items": {"type": "string"}},
        "term_orig":    {"type": "array", "items": {"type": "string"}},
        "legal_areas":  {"type": "array", "items": {"type": "string"}},
    },
    "additionalProperties": False,
}

if bundle is None:
    sub_issue_targets = []
    for issue in sub_issues_en:
        raw = chat_complete(
            TARGETS_PROMPT.format(issue=issue),
            max_tokens=1024,
            json_schema=TARGETS_SCHEMA,
        )
        # Primary: guided decoding gave us clean JSON. Fallback: non-greedy regex.
        t = None
        try:
            t = json.loads(raw.strip())
        except json.JSONDecodeError:
            m = re.search(r"\{.*?\}", raw, re.DOTALL)
            if m:
                try:
                    t = json.loads(m.group(0))
                except json.JSONDecodeError as e:
                    print(f"  WARN: JSONDecodeError on inner object: {e}")
        if t is None or not isinstance(t, dict):
            print(f"  WARN: no JSON object for issue: {issue!r}\n  raw: {raw[:200]}")
            t = {}
        # fill missing keys
        for k in ("statutes", "concepts_en", "term_orig", "legal_areas"):
            t.setdefault(k, [])
        sub_issue_targets.append(t)
        print(f"  issue: {issue[:80]}")
        print(f"    statutes: {t['statutes'][:5]}")
        print(f"    areas:    {t['legal_areas']}")


In [ ]:
# Phase 2e — write bundle and free vLLM
if bundle is None:
    bundle = {
        "qid": QID,
        "parent_query": PARENT_QUERY,
        "parent_gold": PARENT_GOLD,
        "sub_issues": [
            {
                "idx": i,
                "issue_en": sub_issues_en[i],
                "issue_de": sub_issues_de[i],
                "targets": sub_issue_targets[i],
            }
            for i in range(len(sub_issues_en))
        ],
    }
    CKPT_BUNDLE.write_text(json.dumps(bundle, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved bundle -> {CKPT_BUNDLE}")

    # free GPU
    import gc, torch
    del llm
    gc.collect()
    torch.cuda.empty_cache()
    print("vLLM unloaded.")

print(f"\n=== Phase 2 summary ===")
print(f"sub-issues: {len(bundle['sub_issues'])}")


## Phase 3 — 14-channel funnel per (sub-issue × {EN, DE})

For each sub-issue, run the v7.4/v7.5 funnel twice (EN, DE) → union → keep top-200 per sub-issue (deduped on `did`, lowest `rrf_rank` wins).

The funnel module `run_funnel.py` is loaded from `FUNNEL_MODULE/`. It expects `artifacts_root` to be the project root that contains `data/`, `embeddings/`, `artifacts/`, `data_insights/`, `llm_enrichment_output_law_173k/` as siblings.

**Output**: `{QID}_candidates.parquet` with columns `[sub_issue_idx, did, citation, family, rrf_rank, channel_scores, sub_issue_query_lang, ...]` plus per-channel rank/score cols.


In [ ]:
# Phase 3a — skip if already done
if CKPT_CANDIDATES.exists():
    print(f"[skip] {CKPT_CANDIDATES} exists. Delete to re-run Phase 3.")
    cand_df = pd.read_parquet(CKPT_CANDIDATES)
    print(f"  loaded {len(cand_df):,} candidate rows across {cand_df.sub_issue_idx.nunique()} sub-issues")
else:
    cand_df = None


### Phase 3b — funnel module (inlined)

The full v7.4/v7.5 14-channel funnel (1667 lines) is embedded directly into the next cell so this notebook is fully self-contained — no separate Python files to upload alongside.

The module exposes a `run_funnel(query, targets, lang, top_k, artifacts_root, config)` function. Phase 3c calls it per sub-issue.

In [ ]:
"""run_funnel.py — reusable extraction of the v7.4/v7.5 anchor funnel.

Extracted verbatim (channel semantics-preserving) from
    notebooks/03_anchor_funnel_evolution_v4_to_v74/anchor_funnel_v7_4_val001.ipynb

The notebook hardcoded val_001 and the 10 val queries. This module exposes a
single per-(sub-)query entry point:

    run_funnel(query, targets, lang="en", top_k=1000, ...) -> pd.DataFrame

so the decompose-rerank PoC can call the funnel per decomposed sub-query.

Channels implemented (14 — matches `CHANNELS` list in the notebook):
    law_direct_match, court_statute, co_citation, per_area_bedrock,
    statute_backprop, sibling_expansion, graph_forward, graph_reverse,
    graph_2hop (disabled by default), concept_en, term_orig, bm25,
    vector_raw, vector_enriched.

NOTE on "16 channels": the v7.4 notebook has 14 distinct channels. The
v7.5 work added a 15th (concept_cosine) and a HyDE-aspect variant; see
research/v75_chan16_download/. Those are NOT in this module — they are
downstream of this funnel and are wired separately by the dossier code.

Lazy-loading: every heavy artifact (sqlite/parquet/FAISS/embeddings/LLM
enrichment streams) is loaded on first call. Subsequent calls reuse the
cached state. The module does nothing at import time.

Reproduction guarantee: when called with `query=val_001_text` and the
LLM-target dict produced by the notebook (saved at
research/anchor_funnel_val001_v7/targets.json on the notebook run), this
funnel reproduces the notebook's 14-channel CHANNELS list and the RRF-fused
top-1000 within RRF tie-breaking noise. CONFIG defaults below match the
v7.5 values that were live on the notebook's last successful run.
"""

from __future__ import annotations

import json
import math
import re
import sqlite3
import time
from collections import Counter, defaultdict
from pathlib import Path
from typing import Iterable, Mapping

import pandas as pd

# =============================================================================
# 1. PATHS — local default layout (E:/swiss_citation_extraction).
# =============================================================================

DEFAULT_ARTIFACTS_ROOT = Path("e:/swiss_citation_extraction")

# Per-artifact resolution (override via config_paths argument to run_funnel
# or by mutating PATHS in-place before the first call).
PATHS: dict[str, Path] = {
    "val_csv":          DEFAULT_ARTIFACTS_ROOT / "data" / "val.csv",
    "law_llm":          DEFAULT_ARTIFACTS_ROOT / "llm_enrichment_output_law_173k"
                            / "law_llm_descriptors_0000000_all.jsonl",
    "court_v5":         DEFAULT_ARTIFACTS_ROOT / "artifacts"
                            / "court_authority_cards_v5_unified.jsonl",
    "emb_dir":          DEFAULT_ARTIFACTS_ROOT / "embeddings",
    "emb_manifest":     DEFAULT_ARTIFACTS_ROOT / "embeddings"
                            / "qwen3_8b_unified_manifest.parquet",
    "graph_db":         DEFAULT_ARTIFACTS_ROOT / "data_insights"
                            / "citation_graph_db_and_edges"
                            / "citation_graph_extracted.sqlite",
}

# =============================================================================
# 2. CONFIG — copied verbatim from notebook Cell `CONFIG`.
# =============================================================================

CONFIG: dict = {
    "topk_final": 1000,

    # Channel budgets (v7.5 values from notebook Cell 4 source).
    "budget_law_direct":     None,
    "budget_court_statute":  8000,
    "budget_concept":        3000,
    "budget_term":           2500,
    "budget_per_area":       1500,
    "budget_co_citation":    2500,
    "budget_bm25":           2000,
    "budget_vector":         2000,
    "budget_vector_enriched":2000,
    "budget_backprop":       2000,
    "budget_sibling":        5000,
    "budget_graph_forward":  5000,
    "budget_graph_reverse":  3000,
    "enable_graph_2hop":     False,
    "budget_graph_2hop":     1500,

    "rrf_k": 60,
    "guarantee_channels": [
        "law_direct_match",
        "per_area_bedrock",
        "statute_backprop",
        "concept_en",
        "graph_forward",
        "sibling_expansion",
        "term_orig",
    ],
    "guarantee_per_channel": 130,
    "channel_weights": {
        "statute_backprop":  2.5,
        "graph_forward":     2.0,
        "concept_en":        1.8,
        "court_statute":     1.5,
        "per_area_bedrock":  1.5,
        "vector_raw":        1.0,
        "vector_enriched":   1.0,
        "term_orig":         1.2,
        "law_direct_match":  1.2,
        "sibling_expansion": 1.0,
        "bm25":              0.8,
        "co_citation":       0.7,
        "graph_reverse":     0.5,
        "graph_2hop":        0.0,
    },
    "code_family_top_k": 8,

    "per_area_top_n": 1000,
    "co_citation_top_k_per_target":    50,
    "co_citation_min_co_count":        50,
    "co_citation_max_neighbour_count": 50000,
    "concept_substring_top_k": 6,

    "bm25_max_query_terms": 60,
    "bm25_min_token_len":   3,

    "vector_emb_model": "Qwen/Qwen3-Embedding-8B",
    "vector_topk":      800,

    "enhance_top_k_codes":   5,
    "enhance_repeat_count":  5,
    "enhance_min_idf":       1.0,

    "noise_paragraph_roles": {"notification", "header", "empty", "metadata"},

    "lowercase_concepts": True,
    "lowercase_terms":    True,
}

# =============================================================================
# 3. Canonicalization helpers (verbatim from notebook).
# =============================================================================

CODE_ALIAS = {
    "CPP": "StPO", "CP": "StGB", "CC": "ZGB", "CO": "OR",
    "LTF": "BGG", "LACI": "AVIG", "LAA": "UVG",
    "LP": "SchKG", "LDIP": "IPRG", "Cst": "BV", "Cst.": "BV",
    "STPO": "StPO", "OBG": "OR",
}
ART_RE = re.compile(r"art\.?\s*(\d+[a-z]?)", re.I)
CODE_RE = re.compile(r"\b([A-Z][A-Za-z]{1,8}\.?)\b")
CASE_BGE_RE = re.compile(r"BGE\s+(\d+)\s+([IVX]+)\s+(\d+)")
CASE_DOCKET_RE = re.compile(r"\b(\d[A-Z]_\d+/\d{4})\b")
TOKEN_NORM_RE = re.compile(r"\s+")

LEGAL_AREA_DEFAULT_CODE = {
    "criminal law and criminal procedure": "StPO",
    "criminal procedure":                  "StPO",
    "criminal law":                        "StGB",
    "civil law":                           "ZGB",
    "obligations":                         "OR",
    "civil procedure":                     "ZPO",
    "constitutional and public law":       "BV",
    "constitutional law":                  "BV",
    "administrative law":                  "VwVG",
    "social insurance":                    "ATSG",
    "tax law":                             "DBG",
}

_TERM_LEMMA_SUFFIXES = ("en", "es", "em", "er", "e", "n", "s")

_CONCEPT_STOPWORDS_EN = frozenset({
    "a","an","the","of","in","on","at","by","for","with","to","from",
    "and","or","but","is","are","was","were","be","been","being",
    "has","have","had","do","does","did","no","not",
})
_CONCEPT_TOKEN_SPLIT_RE = re.compile(r"[^\w]+", re.UNICODE)


def statute_anchor_canonical(raw: str | None) -> str | None:
    if not raw:
        return None
    s = raw.strip()
    m = ART_RE.search(s)
    if not m:
        return None
    cands = [c.strip(".") for c in CODE_RE.findall(s)
             if c.strip(".") not in ("Art", "Abs", "Ziff", "lit", "let", "al", "Bst")]
    if not cands:
        return None
    code = CODE_ALIAS.get(cands[-1], cands[-1])
    return f"{m.group(1)} {code}"


def article_num(raw: str | None) -> str | None:
    if not raw:
        return None
    m = ART_RE.search(raw.strip())
    return m.group(1) if m else None


def canonicalize_row_anchors(raw_anchors: Iterable[str],
                              legal_area_static: str | None) -> set[str]:
    canons: set[str] = set()
    primary_code = None
    for sa in raw_anchors:
        c = statute_anchor_canonical(sa)
        if c:
            primary_code = c.split()[1]
            break
    fallback = primary_code
    if fallback is None and legal_area_static:
        la = legal_area_static.lower()
        for k, v in LEGAL_AREA_DEFAULT_CODE.items():
            if k in la:
                fallback = v
                break
    for sa in raw_anchors:
        c = statute_anchor_canonical(sa)
        if c:
            canons.add(c); continue
        n = article_num(sa)
        if n and fallback:
            canons.add(f"{n} {fallback}")
    return canons


def case_anchor_canonical(raw: str | None) -> str | None:
    if not raw:
        return None
    s = raw.strip()
    m = CASE_BGE_RE.search(s)
    if m:
        return f"BGE {m.group(1)} {m.group(2)} {m.group(3)}"
    m = CASE_DOCKET_RE.search(s)
    if m:
        return m.group(1)
    return None


def norm_token(s: str | None, lower: bool) -> str | None:
    if not s:
        return None
    s = TOKEN_NORM_RE.sub(" ", s.strip())
    if not s:
        return None
    return s.lower() if lower else s


def term_lemma(tok: str | None) -> str | None:
    if not tok:
        return tok
    cur = tok
    seen = {cur}
    while True:
        changed = False
        for suf in _TERM_LEMMA_SUFFIXES:
            if cur.endswith(suf) and len(cur) - len(suf) >= 4:
                stem = cur[: len(cur) - len(suf)]
                if stem not in seen:
                    cur = stem
                    seen.add(cur)
                    changed = True
                    break
        if not changed:
            break
    return cur


def _take_text(*parts, max_chars: int = 2000) -> str:
    out: list[str] = []
    for p in parts:
        if not p:
            continue
        if isinstance(p, list):
            for x in p:
                if isinstance(x, str):
                    out.append(x)
                elif isinstance(x, dict):
                    for v in x.values():
                        if isinstance(v, str):
                            out.append(v)
        elif isinstance(p, str):
            out.append(p)
    return (" ".join(out))[:max_chars]


# =============================================================================
# 4. State container + lazy loaders.
# =============================================================================

# Use a private module-level singleton so first-call cost is paid once.
_STATE: dict | None = None
_FTS_BUILT = False
_VECTOR_LOADED = False
_GRAPH_LOADED = False


def _build_indexes() -> dict:
    """Stream law_llm + court_v5 jsonl once and build every index the channels
    need. Mirrors notebook Cell 6 (the big indexing cell) exactly.
    """
    print(f"[run_funnel] building corpus indexes — first call only. "
          f"law={PATHS['law_llm']} court={PATHS['court_v5']}")
    t0 = time.time()

    state: dict = {
        "cit_to_doc_ids":       defaultdict(list),
        "doc_meta":             {},
        "idx_law_direct":       defaultdict(set),
        "idx_court_statute":    defaultdict(set),
        "idx_case_anchor":      defaultdict(set),
        "idx_court_base":       defaultdict(set),
        "idx_concept_en":       defaultdict(set),
        "idx_term_orig":        defaultdict(set),
        "idx_term_lemma":       defaultdict(set),
        "term_orig_keys":       set(),
        "legal_area_per_doc":   {},
        "search_text":          {},
        "co_citation_pairs":    Counter(),
        "tlf":                  defaultdict(Counter),
        "token_doc_count":      Counter(),
        "doc_statute_anchors":  {},
        "doc_language":         {},
    }
    DOC_ID_LAW   = lambda i: f"law:{i}"
    DOC_ID_COURT = lambda i: f"court:{i}"

    # --- Law stream ----------------------------------------------------------
    n_law = 0
    with open(PATHS["law_llm"], encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
            except Exception:
                continue
            cit = obj.get("citation", "")
            if not cit:
                continue
            did = DOC_ID_LAW(n_law)
            state["cit_to_doc_ids"][cit].append(did)
            state["doc_meta"][did] = {
                "citation": cit, "family": "law", "court_base": None,
                "paragraph_role": None, "is_notification_paragraph": False,
            }
            canon = statute_anchor_canonical(cit)
            if canon:
                state["idx_law_direct"][canon].add(did)

            enr = obj.get("llm_enrichment") or {}
            terms_de: list[str] = []
            terms_en: list[str] = []
            for t in enr.get("terms_de_to_en") or []:
                if isinstance(t, dict):
                    de = norm_token(t.get("de", ""), CONFIG["lowercase_terms"])
                    en = norm_token(t.get("en", ""), CONFIG["lowercase_terms"])
                    if de:
                        state["idx_term_orig"][de].add(did)
                        terms_de.append(de)
                        state["term_orig_keys"].add(de)
                        _lem = term_lemma(de)
                        if _lem and _lem != de:
                            state["idx_term_lemma"][_lem].add(did)
                        state["idx_term_lemma"][de].add(did)
                    if en:
                        state["idx_concept_en"][en].add(did)
                        terms_en.append(en)
            for c in enr.get("concepts_en") or []:
                tok = norm_token(c, CONFIG["lowercase_concepts"])
                if tok:
                    state["idx_concept_en"][tok].add(did)
            state["search_text"][did] = _take_text(
                cit, enr.get("english_summary", ""), enr.get("legal_rule", ""),
                enr.get("legal_question", ""), enr.get("applicability_conditions"),
                enr.get("concepts_en"), terms_de, terms_en,
            )
            state["legal_area_per_doc"][did] = "law"
            state["doc_language"][did] = (obj.get("language") or "de").lower()

            if canon and " " in canon:
                row_code = canon.split()[1].lower()
                row_text = state["search_text"][did].lower()
                row_tokens: set[str] = set()
                for tok in re.split(r"[^\w\d]+", row_text, flags=re.UNICODE):
                    if len(tok) >= 3:
                        row_tokens.add(tok)
                for tok in row_tokens:
                    state["tlf"][tok][row_code] += 1
                    state["token_doc_count"][tok] += 1
            n_law += 1
    print(f"[run_funnel] law: {n_law:,} rows in {time.time()-t0:.1f}s")

    # --- Court stream --------------------------------------------------------
    t1 = time.time()
    n_court = 0
    with open(PATHS["court_v5"], encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
            except Exception:
                continue
            cit = obj.get("citation", "")
            if not cit:
                continue
            did = DOC_ID_COURT(n_court)
            state["cit_to_doc_ids"][cit].append(did)
            cb = obj.get("court_base") or ""
            rag = obj.get("rag_enrichment") or {}
            legal_area_static = obj.get("legal_area_static") or rag.get("legal_area") or ""
            state["doc_meta"][did] = {
                "citation": cit, "family": "court", "court_base": cb,
                "paragraph_role": rag.get("paragraph_role"),
                "is_notification_paragraph": bool(obj.get("is_notification_paragraph")),
            }
            state["legal_area_per_doc"][did] = (legal_area_static or "").lower()
            _lang_raw = (obj.get("language") or "").lower()
            state["doc_language"][did] = (_lang_raw if _lang_raw in ("de", "fr", "it", "en") else "en")

            if cb:
                state["idx_court_base"][cb].add(did)
                cb_canon = case_anchor_canonical(cb)
                if cb_canon:
                    state["idx_case_anchor"][cb_canon].add(did)

            row_canons = canonicalize_row_anchors(rag.get("statute_anchors") or [], legal_area_static)
            for canon in row_canons:
                state["idx_court_statute"][canon].add(did)
            if row_canons:
                state["doc_statute_anchors"][did] = row_canons
            rc = sorted(row_canons)
            for i in range(len(rc)):
                for j in range(i+1, len(rc)):
                    state["co_citation_pairs"][(rc[i], rc[j])] += 1

            for ca in rag.get("case_anchors") or []:
                canon = case_anchor_canonical(ca)
                if canon:
                    state["idx_case_anchor"][canon].add(did)
            for c in rag.get("concepts_en") or []:
                tok = norm_token(c, CONFIG["lowercase_concepts"])
                if tok:
                    state["idx_concept_en"][tok].add(did)
            for t in rag.get("terms_original") or []:
                tok = norm_token(t, CONFIG["lowercase_terms"])
                if tok:
                    state["idx_term_orig"][tok].add(did)
                    state["term_orig_keys"].add(tok)
                    _lem = term_lemma(tok)
                    if _lem and _lem != tok:
                        state["idx_term_lemma"][_lem].add(did)
                    state["idx_term_lemma"][tok].add(did)

            state["search_text"][did] = _take_text(
                cit, obj.get("text_excerpt_original", ""),
                rag.get("concepts_en"), rag.get("terms_original"),
                rag.get("micro_topic", ""), rag.get("topic", ""), rag.get("subtopic", ""),
                rag.get("statute_anchors"),
            )
            n_court += 1
            if n_court % 500_000 == 0:
                print(f"[run_funnel] court progress: {n_court:,} ({time.time()-t1:.1f}s)")
    print(f"[run_funnel] court: {n_court:,} rows in {time.time()-t1:.1f}s")
    print(f"[run_funnel] total docs: {len(state['doc_meta']):,}")

    # --- Derived counts -----------------------------------------------------
    state["idx_court_statute_count"] = {
        canon: len(s) for canon, s in state["idx_court_statute"].items()
    }

    # --- Per-area bedrock + canon counts + co-cite neighbours + code pairs --
    per_area_canon_count: dict[str, Counter] = defaultdict(Counter)
    for did, area in state["legal_area_per_doc"].items():
        if not area or area == "law":
            continue
        canons = state["doc_statute_anchors"].get(did, ())
        for canon in canons:
            per_area_canon_count[area][canon] += 1
    state["per_area_canon_count"] = per_area_canon_count

    canon_count: Counter = Counter()
    for did, canons in state["doc_statute_anchors"].items():
        for c in canons:
            canon_count[c] += 1
    state["canon_count"] = canon_count

    co_neighbours_raw: dict[str, list[tuple[str, int]]] = defaultdict(list)
    for (a, b), cnt in state["co_citation_pairs"].items():
        if cnt < CONFIG["co_citation_min_co_count"]:
            continue
        co_neighbours_raw[a].append((b, cnt))
        co_neighbours_raw[b].append((a, cnt))
    co_neighbours = {
        src: sorted(
            ((nb, n) for nb, n in nbs
             if canon_count[nb] <= CONFIG["co_citation_max_neighbour_count"]),
            key=lambda x: -x[1],
        )[: CONFIG["co_citation_top_k_per_target"] * 2]
        for src, nbs in co_neighbours_raw.items()
    }
    state["co_neighbours"] = co_neighbours

    code_pair_count: Counter = Counter()
    for (a, b), n in state["co_citation_pairs"].items():
        ca = a.split()[1] if " " in a else None
        cb = b.split()[1] if " " in b else None
        if ca and cb and ca != cb:
            code_pair_count[(ca, cb)] += n
            code_pair_count[(cb, ca)] += n
    state["code_pair_count"] = code_pair_count

    return state


def _get_state() -> dict:
    global _STATE
    if _STATE is None:
        _STATE = _build_indexes()
    return _STATE


# =============================================================================
# 5. FTS5 — per-language in-memory build. Lazy.
# =============================================================================

_LANGS = ("de", "fr", "it", "en")
_FTS_BY_LANG: dict[str, sqlite3.Connection] = {}
_LANG_DOC_COUNT: dict[str, int] = {}


def _build_fts() -> None:
    global _FTS_BUILT, _FTS_BY_LANG, _LANG_DOC_COUNT
    if _FTS_BUILT:
        return
    st = _get_state()
    print(f"[run_funnel] building per-language FTS5 over {len(st['search_text']):,} docs...")
    t = time.time()
    for L in _LANGS:
        conn = sqlite3.connect(":memory:")
        conn.execute("PRAGMA journal_mode = MEMORY")
        conn.execute("PRAGMA synchronous = OFF")
        conn.execute(
            "CREATE VIRTUAL TABLE docs USING fts5(did UNINDEXED, body, "
            "tokenize = 'unicode61 remove_diacritics 2')"
        )
        _FTS_BY_LANG[L] = conn
        _LANG_DOC_COUNT[L] = 0

    batches: dict[str, list[tuple[str, str]]] = {L: [] for L in _LANGS}
    for did, txt in st["search_text"].items():
        L = st["doc_language"].get(did, "en")
        if L not in _FTS_BY_LANG:
            L = "en"
        batches[L].append((did, txt))
        _LANG_DOC_COUNT[L] += 1
        if len(batches[L]) >= 50000:
            _FTS_BY_LANG[L].executemany(
                "INSERT INTO docs(did, body) VALUES (?, ?)", batches[L]
            )
            batches[L].clear()
    for L in _LANGS:
        if batches[L]:
            _FTS_BY_LANG[L].executemany(
                "INSERT INTO docs(did, body) VALUES (?, ?)", batches[L]
            )
        _FTS_BY_LANG[L].commit()
    print(f"[run_funnel] FTS5 built in {time.time()-t:.1f}s; "
          f"{', '.join(f'{L}={_LANG_DOC_COUNT[L]:,}' for L in _LANGS)}")
    _FTS_BUILT = True


# =============================================================================
# 6. enhance() — corpus-derived BM25 lexicon expansion.
# =============================================================================

def _enhance_codes(text: str) -> list[str]:
    st = _get_state()
    text_lc = (text or "").lower()
    tokens: set[str] = set()
    for tok in re.split(r"[^\w\d]+", text_lc, flags=re.UNICODE):
        if len(tok) >= CONFIG["bm25_min_token_len"]:
            tokens.add(tok)
    code_score: Counter = Counter()
    for tok in tokens:
        if tok not in st["tlf"]:
            continue
        n_docs = max(1, st["token_doc_count"][tok])
        idf = math.log(1 + (max(1, len(st["search_text"])) / n_docs))
        if idf < CONFIG["enhance_min_idf"]:
            continue
        for code, cnt in st["tlf"][tok].most_common():
            code_score[code] += cnt * idf
    return [c for c, _ in code_score.most_common(CONFIG["enhance_top_k_codes"])]


def _build_lang_query(english_query: str, targets: Mapping, lang: str) -> str:
    parts = [english_query or ""]
    targets = targets or {}
    concept_en = list(targets.get("concept_targets_en") or [])
    term_de = list(targets.get("term_targets_de") or [])
    term_fr = list(targets.get("term_targets_fr") or [])
    if lang == "de":
        parts.extend(term_de); parts.extend(concept_en)
    elif lang == "fr":
        parts.extend(term_fr); parts.extend(concept_en)
    elif lang == "it":
        parts.extend(term_de); parts.extend(term_fr); parts.extend(concept_en)
    else:
        parts.extend(concept_en)
    return " ".join(p for p in parts if p)


def _split_budget(k_total: int) -> dict[str, int]:
    total_docs = sum(max(1, _LANG_DOC_COUNT[L]) for L in _LANGS)
    floor = max(1, k_total // 16)
    raw = {L: max(floor, int(round(k_total * _LANG_DOC_COUNT[L] / total_docs)))
           for L in _LANGS}
    over = sum(raw.values()) - k_total
    if over > 0:
        for L in sorted(_LANGS, key=lambda x: -raw[x]):
            take = min(over, raw[L] - floor)
            if take <= 0:
                continue
            raw[L] -= take
            over -= take
            if over <= 0:
                break
    return raw


def bm25_search_multilang(query: str, targets: Mapping,
                          k_total: int) -> list[tuple[str, float]]:
    """Multi-language BM25. Verbatim from notebook bm25_search_multilang."""
    _build_fts()
    if not _FTS_BY_LANG:
        return []
    budgets = _split_budget(k_total)
    boosted = _enhance_codes(query or "")
    boost_str = " ".join(c * CONFIG["enhance_repeat_count"] for c in boosted)

    merged: dict[str, float] = {}
    for L in _LANGS:
        if _LANG_DOC_COUNT[L] == 0:
            continue
        per_lang_query = _build_lang_query(query, targets, L)
        if boost_str:
            per_lang_query = per_lang_query + " " + boost_str
        fts_q: list[str] = []
        seen: set[str] = set()
        for tok in re.split(r"[^\w\d]+", per_lang_query, flags=re.UNICODE):
            if len(tok) < CONFIG["bm25_min_token_len"]:
                continue
            tl = tok.lower()
            if tl in seen:
                continue
            seen.add(tl); fts_q.append(tok)
            if len(fts_q) >= CONFIG["bm25_max_query_terms"]:
                break
        if not fts_q:
            continue
        fts_query = " OR ".join(f'"{t}"' for t in fts_q)
        denom = math.log(_LANG_DOC_COUNT[L] + math.e)
        try:
            rows = _FTS_BY_LANG[L].execute(
                "SELECT did, bm25(docs) FROM docs WHERE docs MATCH ? "
                "ORDER BY bm25(docs) LIMIT ?",
                (fts_query, budgets[L]),
            ).fetchall()
        except sqlite3.OperationalError as e:
            print(f"[run_funnel] bm25[{L}] skipped: {e}")
            continue
        for did, raw_score in rows:
            norm = (-raw_score) / denom
            prev = merged.get(did)
            if prev is None or norm > prev:
                merged[did] = norm
    if not merged:
        return []
    return sorted(merged.items(), key=lambda kv: -kv[1])[:k_total]


# =============================================================================
# 7. Graph load (lazy).
# =============================================================================

def _load_graph() -> None:
    global _GRAPH_LOADED
    if _GRAPH_LOADED:
        return
    st = _get_state()
    st["idx_graph_out"] = defaultdict(list)
    st["idx_graph_in"]  = defaultdict(list)
    st["idx_judgment_importance"] = {}

    if not PATHS["graph_db"].exists():
        print(f"[run_funnel] graph db missing at {PATHS['graph_db']}; graph channels empty.")
        _GRAPH_LOADED = True
        return

    print(f"[run_funnel] loading graph from {PATHS['graph_db']}...")
    t = time.time()
    cit_to_did = {cit: dids[0] for cit, dids in st["cit_to_doc_ids"].items() if dids}
    conn = sqlite3.connect(str(PATHS["graph_db"]))
    n_loaded = 0; n_skipped = 0
    for src, tgt in conn.execute(
        "SELECT source, target FROM edges WHERE dataset='court_considerations'"
    ):
        s = cit_to_did.get(src)
        t2 = cit_to_did.get(tgt)
        if s is None or t2 is None:
            n_skipped += 1; continue
        if s == t2:
            continue
        st["idx_graph_out"][s].append(t2)
        st["idx_graph_in"][t2].append(s)
        n_loaded += 1
    conn.close()
    print(f"[run_funnel] graph: {n_loaded:,} edges loaded, {n_skipped:,} skipped, "
          f"{time.time()-t:.1f}s")

    for cb, dids in st["idx_court_base"].items():
        imp = 0
        for d in dids:
            imp += len(st["idx_graph_in"].get(d, ()))
        st["idx_judgment_importance"][cb] = imp
    _GRAPH_LOADED = True


# =============================================================================
# 8. Vector setup (lazy; optional — silently disabled if torch / chunks absent).
# =============================================================================

_VECTOR: dict = {"ok": False, "E_GPU": None, "row_for_did": None,
                 "my_did_for_row": None, "device": None}
_EMB_MODEL = None


def _load_vector() -> None:
    """Lazy-load the corpus-wide Qwen3-Embedding-8B matrix to GPU.
    Mirrors notebook Cell 12. Silently skips if torch / chunks / manifest
    aren't available — the channel functions then return [].
    """
    global _VECTOR_LOADED
    if _VECTOR_LOADED:
        return
    _VECTOR_LOADED = True

    try:
        import torch  # noqa
        import numpy as np  # noqa
    except Exception as e:
        print(f"[run_funnel] vector disabled (torch/numpy not importable): {e}")
        return
    if not PATHS["emb_manifest"].exists():
        print(f"[run_funnel] vector disabled: emb_manifest missing at {PATHS['emb_manifest']}")
        return
    chunks = sorted(PATHS["emb_dir"].glob("qwen3_8b_unified_chunk*.npy"))
    if not chunks:
        print(f"[run_funnel] vector disabled: no chunk*.npy in {PATHS['emb_dir']}")
        return

    import torch
    import numpy as np
    st = _get_state()
    print(f"[run_funnel] loading manifest + {len(chunks)} chunks to GPU/CPU...")
    t = time.time()
    man = pd.read_parquet(PATHS["emb_manifest"])

    row_for_did: dict[str, int] = {}
    my_did_for_row: list[str | None] = [None] * len(man)
    cit_to_dids = st["cit_to_doc_ids"]
    doc_meta = st["doc_meta"]
    for _, r in man.iterrows():
        cit = r.get("citation") or ""
        fam = r.get("family") or ""
        ridx = int(r.get("row_index", -1))
        if ridx < 0:
            continue
        for d in cit_to_dids.get(cit, []):
            if doc_meta.get(d, {}).get("family") == fam:
                row_for_did[d] = ridx
                if ridx < len(my_did_for_row):
                    my_did_for_row[ridx] = d
                break
    print(f"[run_funnel] manifest mapped in {time.time()-t:.1f}s")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    arrs: list = []
    for cp in chunks:
        arrs.append(torch.from_numpy(np.load(cp)).to(device, non_blocking=True))
    E = torch.cat(arrs, dim=0)
    del arrs
    print(f"[run_funnel] E shape={tuple(E.shape)} dtype={E.dtype} on {device}; "
          f"{time.time()-t:.1f}s total")
    _VECTOR["ok"] = True
    _VECTOR["E_GPU"] = E
    _VECTOR["row_for_did"] = row_for_did
    _VECTOR["my_did_for_row"] = my_did_for_row
    _VECTOR["device"] = device


def _vector_search(q_emb, k: int) -> list[tuple[str, float]]:
    if not _VECTOR["ok"] or _VECTOR["E_GPU"] is None:
        return []
    import torch
    E = _VECTOR["E_GPU"]
    with torch.no_grad():
        q = q_emb.to(E.device, dtype=E.dtype)
        q = q / (q.norm(dim=-1, keepdim=True) + 1e-9)
        scores = E @ q
        top_v, top_i = torch.topk(scores, k=min(k, scores.shape[0]))
    out: list[tuple[str, float]] = []
    my_did_for_row = _VECTOR["my_did_for_row"]
    for s, i in zip(top_v.cpu().tolist(), top_i.cpu().tolist()):
        d = my_did_for_row[i]
        if d is not None:
            out.append((d, float(s)))
    return out


def _encode_query(text: str):
    """Lazy-load Qwen3-Embedding-8B sentence-transformer and embed `text`."""
    global _EMB_MODEL
    if _EMB_MODEL is None:
        from sentence_transformers import SentenceTransformer
        print(f"[run_funnel] loading {CONFIG['vector_emb_model']}...")
        _EMB_MODEL = SentenceTransformer(CONFIG["vector_emb_model"])
    return _EMB_MODEL.encode(
        [text], prompt_name="query",
        convert_to_tensor=True, normalize_embeddings=True,
    )[0]


# =============================================================================
# 9. Channel functions — verbatim semantics from notebook Cell 16.
# =============================================================================

def channel_law_direct(canon_set, idx, budget):
    counter = Counter()
    for canon in canon_set:
        for did in idx[canon]:
            counter[did] += 1
    items = counter.most_common()
    return items if budget is None else items[:budget]


def channel_court_statute(statute_canons, idx, idx_count=None, doc_meta=None, budget=None):
    if idx_count is None:
        idx_count = {c: len(idx[c]) for c in statute_canons if c in idx}
    _ROLE_W = {
        "legal_standard": 1.5, "reasoning": 1.5,
        "application":    1.5, "holding":   1.5,
        "facts":              1.0, "procedural_history": 1.0,
        "citation":           1.0, "neutral_default":    1.0,
        "costs":        0.6, "disposition": 0.6, "notification": 0.6,
        "neutral":      0.4,
    }
    per_doc: dict[str, list[float]] = {}
    for canon in statute_canons:
        dids = idx.get(canon)
        if not dids:
            continue
        cnt = idx_count.get(canon, len(dids))
        w_canon = 1.0 / math.log(2 + cnt)
        for did in dids:
            slot = per_doc.get(did)
            if slot is None:
                per_doc[did] = [1, w_canon]
            else:
                slot[0] += 1
                slot[1] += w_canon
    if not per_doc:
        return []
    scored = []
    for did, (n_matches, base_w) in per_doc.items():
        score = base_w * (1.0 + 0.3 * (n_matches - 1))
        if doc_meta is not None:
            meta = doc_meta.get(did) or {}
            role = meta.get("paragraph_role")
            if role is None or role == "":
                rw = 0.4
            else:
                rw = _ROLE_W.get(role, 1.0)
            score *= rw
        scored.append((did, float(score)))
    scored.sort(key=lambda x: (-x[1], x[0]))
    return scored if budget is None else scored[:budget]


_SUBSTANTIVE_ROLES = {"reasoning", "legal_standard", "application", "holding"}


def _judgment_factor(cb, idx_judgment_importance):
    imp = idx_judgment_importance.get(cb, 0) if cb else 0
    if imp <= 0:
        return 1.0
    return math.sqrt(1.0 + math.log(1.0 + float(imp)))


def _role_boost(doc_meta, did):
    m = doc_meta.get(did) or {}
    role = (m.get("paragraph_role") or "").strip().lower()
    return 1.5 if role in _SUBSTANTIVE_ROLES else 1.0


def channel_sibling(seed_doc_ids, idx_court_base, idx_judgment_importance,
                    doc_meta, budget):
    seed_count = Counter()
    for did in seed_doc_ids:
        m = doc_meta.get(did) or {}
        cb = m.get("court_base")
        if cb:
            seed_count[cb] += 1
    seed_set = set(seed_doc_ids)
    scored: dict[str, float] = {}
    for cb, cnt in seed_count.items():
        factor = _judgment_factor(cb, idx_judgment_importance)
        for s in idx_court_base.get(cb, ()):
            if s in seed_set:
                continue
            sc = float(cnt) * factor * _role_boost(doc_meta, s)
            if sc > scored.get(s, 0.0):
                scored[s] = sc
    items = sorted(scored.items(), key=lambda kv: (-kv[1], kv[0]))
    return items if budget is None else items[:budget]


def channel_graph_forward(seed_doc_ids, idx_graph_out, idx_judgment_importance,
                          doc_meta, budget):
    seed_set = set(seed_doc_ids)
    edge_count = Counter()
    for did in seed_doc_ids:
        for t in idx_graph_out.get(did, ()):
            edge_count[t] += 1
    for d in seed_set:
        edge_count.pop(d, None)
    scored: dict[str, float] = {}
    for t, cnt in edge_count.items():
        m = doc_meta.get(t) or {}
        cb_of_t = m.get("court_base")
        factor = _judgment_factor(cb_of_t, idx_judgment_importance)
        scored[t] = float(cnt) * factor * _role_boost(doc_meta, t)
    items = sorted(scored.items(), key=lambda kv: (-kv[1], kv[0]))
    return items if budget is None else items[:budget]


def channel_graph_reverse(seed_doc_ids, idx_graph_in, idx_judgment_importance,
                          doc_meta, budget, landmark_imp_threshold=5):
    seed_set = set(seed_doc_ids)
    scored: dict[str, float] = {}
    for did in seed_doc_ids:
        m = doc_meta.get(did) or {}
        cb_seed = m.get("court_base")
        imp_seed = idx_judgment_importance.get(cb_seed, 0) if cb_seed else 0
        if imp_seed < landmark_imp_threshold:
            continue
        factor = _judgment_factor(cb_seed, idx_judgment_importance)
        for s in idx_graph_in.get(did, ()):
            if s in seed_set:
                continue
            inc = factor * _role_boost(doc_meta, s)
            scored[s] = scored.get(s, 0.0) + inc
    items = sorted(scored.items(), key=lambda kv: (-kv[1], kv[0]))
    return items if budget is None else items[:budget]


def channel_graph_2hop(seed_doc_ids, idx_graph_out, budget):
    seed_set = set(seed_doc_ids)
    intermediate: set[str] = set()
    for did in seed_doc_ids:
        intermediate.update(idx_graph_out.get(did, ()))
    intermediate -= seed_set
    counter = Counter()
    for x in intermediate:
        for t in idx_graph_out.get(x, ()):
            if t in seed_set:
                continue
            counter[t] += 1
    for d in intermediate:
        counter.pop(d, None)
    return counter.most_common(budget)


def _concept_tokens(s):
    if not s:
        return []
    return [t for t in _CONCEPT_TOKEN_SPLIT_RE.split(s.lower()) if t]


def _concept_meaningful_tokens(s):
    return [t for t in _concept_tokens(s) if t not in _CONCEPT_STOPWORDS_EN]


def expand_concepts_weighted(llm_concepts, corpus_concept_keys, top_k=25):
    W_EXACT = 1.0
    W_SUBSTR_MAX = 0.85
    keys = list(corpus_concept_keys)
    corpus_meaningful: dict[str, tuple[set[str], int]] = {}
    for k in keys:
        m = _concept_meaningful_tokens(k)
        if m:
            corpus_meaningful[k] = (set(m), len(m))

    best_weight: dict[str, float] = {}
    for raw in llm_concepts or []:
        c = (raw or "").lower().strip()
        if not c or len(c) < 4:
            continue
        c_meaningful = _concept_meaningful_tokens(c)
        if not c_meaningful:
            continue
        c_set = set(c_meaningful)
        c_len = len(c_meaningful)
        c_chars = len(c)

        per_query: list[tuple[float, int, str]] = []
        if c in corpus_concept_keys:
            per_query.append((W_EXACT, 0, c))
        for cv in keys:
            if cv == c:
                continue
            if c in cv or cv in c:
                cv_info = corpus_meaningful.get(cv)
                if not cv_info:
                    continue
                cv_set, cv_len = cv_info
                if not (c_set & cv_set):
                    continue
                shorter = min(c_chars, len(cv))
                longer = max(c_chars, len(cv))
                if longer <= 0:
                    continue
                w = W_SUBSTR_MAX * (shorter / longer)
                per_query.append((w, abs(len(cv) - c_chars), cv))
        for cv, (cv_set, cv_len) in corpus_meaningful.items():
            if cv == c:
                continue
            shared = c_set & cv_set
            if not shared:
                continue
            denom = max(c_len, cv_len)
            if denom <= 0:
                continue
            w = len(shared) / denom
            per_query.append((w, abs(cv_len - c_len), cv))
        local_best: dict[str, tuple[float, int]] = {}
        for w, ld, cv in per_query:
            cur = local_best.get(cv)
            if cur is None or w > cur[0] or (w == cur[0] and ld < cur[1]):
                local_best[cv] = (w, ld)
        ranked = sorted(local_best.items(),
                        key=lambda kv: (-kv[1][0], kv[1][1], kv[0]))
        for cv, (w, _) in ranked[:top_k]:
            prior = best_weight.get(cv, 0.0)
            if w > prior:
                best_weight[cv] = w
    return sorted(best_weight.items(), key=lambda kv: (-kv[1], kv[0]))


def channel_concept(expanded_weighted, idx, budget):
    scores: dict[str, float] = {}
    for item in expanded_weighted or []:
        if isinstance(item, tuple):
            tok, w = item
        else:
            tok, w = item, 1.0
        if not tok:
            continue
        for did in idx.get(tok, ()):
            scores[did] = scores.get(did, 0.0) + float(w)
    if not scores:
        return []
    items = sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]))
    return items if budget is None else items[:budget]


def channel_term(targets, idx, idx_lemma, budget,
                 corpus_keys=None, lemma_score=0.7, min_substring_len=4):
    if corpus_keys is None:
        corpus_keys = list(idx.keys())
    else:
        corpus_keys = list(corpus_keys)
    q_terms: list[str] = []
    for key in ("term_targets_de", "term_targets_fr"):
        for raw in targets.get(key, []) or []:
            tok = norm_token(raw, CONFIG["lowercase_terms"])
            if tok:
                q_terms.append(tok)
    if not q_terms:
        return []
    score: Counter = Counter()
    for Q in q_terms:
        Q_lemma = term_lemma(Q)
        Q_len = len(Q)
        per_q: dict[str, float] = {}
        def _bump(did, s):
            if s > per_q.get(did, 0.0):
                per_q[did] = s
        if Q in idx:
            for did in idx[Q]:
                _bump(did, 1.0)
        if Q_lemma and Q_lemma in idx_lemma:
            for did in idx_lemma[Q_lemma]:
                _bump(did, lemma_score)
        if Q_len >= 1:
            for T in corpus_keys:
                if T == Q:
                    continue
                T_len = len(T)
                s = 0.0
                if Q in T:
                    s = Q_len / T_len
                elif T_len >= min_substring_len and T in Q:
                    s = T_len / Q_len
                if s <= 0.0:
                    continue
                for did in idx.get(T, ()):
                    _bump(did, s)
        for did, s in per_q.items():
            score[did] += s
    items = sorted(score.items(), key=lambda kv: (-kv[1], kv[0]))
    return items if budget is None else items[:budget]


def channel_per_area_bedrock(legal_area_keywords, statute_target_codes,
                              per_area_canon_count, idx_law_direct, budget):
    if not legal_area_keywords:
        return []
    keys = [k.lower() for k in legal_area_keywords]
    selected_areas: set[str] = set()
    for area in per_area_canon_count.keys():
        for k in keys:
            if k in area:
                selected_areas.add(area); break
    if not selected_areas:
        return []
    canon_score: Counter = Counter()
    for area in selected_areas:
        for canon, n in per_area_canon_count[area].most_common(CONFIG["per_area_top_n"]):
            canon_score[canon] = max(canon_score[canon], n)
    out: list[tuple[str, int]] = []
    seen: set[str] = set()
    for canon, _ in canon_score.most_common():
        canon_code = canon.split()[1] if canon and " " in canon else None
        if statute_target_codes and canon_code not in statute_target_codes:
            continue
        for did in idx_law_direct.get(canon, set()):
            if did not in seen:
                out.append((did, canon_score[canon])); seen.add(did)
        if len(out) >= budget:
            break
    return out[:budget]


def channel_statute_backprop(seed_court_dids, doc_statute_anchors,
                              idx_law_direct, idx_court_statute_count, budget):
    canon_to_courts: dict[str, set[str]] = defaultdict(set)
    for did in seed_court_dids:
        for canon in doc_statute_anchors.get(did, set()):
            canon_to_courts[canon].add(did)
    counter: Counter = Counter()
    for canon, court_set in canon_to_courts.items():
        n_caught = len(court_set)
        global_ct = idx_court_statute_count.get(canon, n_caught)
        score = n_caught * (1.0 / math.log(2 + global_ct))
        for law_did in idx_law_direct.get(canon, set()):
            if counter[law_did] < score:
                counter[law_did] = score
    return counter.most_common(budget)


def channel_co_citation(targets, co_neighbours, idx_law_direct,
                         idx_court_statute, canon_count, budget):
    counter: Counter = Counter()
    for raw in targets.get("statute_targets", []) or []:
        canon = statute_anchor_canonical(raw)
        if not canon:
            continue
        for nb, n in co_neighbours.get(canon, []):
            spec = 1.0 / math.log(2 + canon_count.get(nb, 1))
            score = n * spec
            for did in idx_law_direct.get(nb, set()):
                if counter[did] < score:
                    counter[did] = score
            for did in idx_court_statute.get(nb, set()):
                if counter[did] < score:
                    counter[did] = score
    return counter.most_common(budget)


# =============================================================================
# 10. RRF fusion + negative gate + round-robin guarantee (notebook Cell 18).
# =============================================================================

def rrf_fuse(channels, k, weights=None):
    if weights is None:
        weights = {}
    score: dict[str, float] = defaultdict(float)
    for name, hits in channels:
        w = weights.get(name, 1.0)
        if w == 0:
            continue
        for rank, (did, _) in enumerate(hits):
            score[did] += w / (k + rank + 1)
    return score


_RRF_SUBSTANTIVE = {
    "facts", "reasoning", "legal_standard", "application",
    "holding", "citation", "procedural_history",
}


def apply_neg_gate(doc_ids, doc_meta, noise_roles):
    keep = []
    for did in doc_ids:
        m = doc_meta.get(did) or {}
        pr = (m.get("paragraph_role") or "").lower()
        if pr in _RRF_SUBSTANTIVE:
            keep.append(did); continue
        if m.get("is_notification_paragraph"):
            continue
        if pr in noise_roles:
            continue
        keep.append(did)
    return keep


def round_robin_guarantee(channels_by_name, guarantee_channel_names,
                           per_channel_cap, total_cap):
    iters = {cn: iter(channels_by_name.get(cn, [])) for cn in guarantee_channel_names}
    counts = {cn: 0 for cn in guarantee_channel_names}
    out: list[str] = []
    seen: set[str] = set()
    while iters and len(out) < total_cap:
        exhausted = []
        for cn in list(iters.keys()):
            if counts[cn] >= per_channel_cap:
                exhausted.append(cn); continue
            try:
                did, _ = next(iters[cn])
                while did in seen:
                    did, _ = next(iters[cn])
                out.append(did); seen.add(did); counts[cn] += 1
                if len(out) >= total_cap:
                    break
            except StopIteration:
                exhausted.append(cn)
        for cn in exhausted:
            if cn in iters:
                del iters[cn]
    return out


# =============================================================================
# 11. Public entry point — run_funnel.
# =============================================================================

def run_funnel(
    query: str,
    targets: Mapping,
    lang: str = "en",
    top_k: int = 1000,
    artifacts_root: Path = DEFAULT_ARTIFACTS_ROOT,
    config: Mapping | None = None,
) -> pd.DataFrame:
    """Run the 14-channel v7.4/v7.5 anchor funnel for a single (sub-)query.

    Args:
        query: the (sub-)query text (English by default).
        targets: dict with keys:
            statute_targets:    list of "Art. N CODE" strings
            concept_targets_en: list of English concept phrases
            term_targets_de:    list of German term strings
            term_targets_fr:    list of French term strings
            legal_area_keywords:list of legal-area phrases
            case_targets:       list of case anchor strings (unused by funnel
                                directly; v5 measured BGE-number hallucination
                                so case anchors were dropped as a channel)
            Also accepts the caller's preferred keys mapped from the
            user's spec: "statutes"->statute_targets,
            "concepts_en"->concept_targets_en, "term_orig"->term_targets_de,
            "legal_areas"->legal_area_keywords.
        lang: "en", "de", "fr", or "it". Controls BM25 language routing
            (per-language FTS5 + per-language target injection) and which
            vector_enriched embedding is built. The Qwen3-Embedding-8B
            model is multilingual so the dense channel still works across
            languages; the `lang` parameter only changes which terms get
            appended to the enriched query string.
            NOTE: vector_raw uses `query` as-is regardless of lang.
        top_k: final pool size (default 1000).
        artifacts_root: defaults to e:/swiss_citation_extraction. If
            provided, rewrites PATHS in-place before the first call.
        config: optional dict of CONFIG overrides (e.g. channel_weights,
            rrf_k, budget_*).

    Returns:
        pd.DataFrame with columns:
            did            (str)
            citation       (str)
            family         ("law" | "court")
            rrf_rank       (int — 0-based final rank)
            rrf_score      (float — fused RRF score, weighted)
            channel_scores (dict[str, float | None])
            rank_<ch>      (int | None — per-channel rank, 0-based)
            score_<ch>     (float | None — per-channel raw score)
            in_guarantee   (bool)
        Length <= top_k.
    """
    global PATHS

    def _first_existing(*candidates: Path) -> Path:
        """Return the first existing candidate. If none exist, return the first
        one anyway (downstream open() will raise with the chosen path so the
        user can see what was attempted). Walk every candidate even after the
        first miss so a layout typo is obvious from the error message."""
        for c in candidates:
            if c.exists():
                return c
        return candidates[0]

    if artifacts_root != DEFAULT_ARTIFACTS_ROOT:
        # Try the local layout first (matches DEFAULT_ARTIFACTS_ROOT), then the
        # Drive layout we extracted from older Colab notebooks. Both end up at
        # the same logical file under different parent subdirs.
        root = artifacts_root
        PATHS["val_csv"] = _first_existing(
            root / "data" / "val.csv",
        )
        PATHS["law_llm"] = _first_existing(
            # local
            root / "llm_enrichment_output_law_173k" / "law_llm_descriptors_0000000_all.jsonl",
            # Drive (per _drive_pull notebook scan: 'data/checkpoints/...')
            root / "data" / "checkpoints" / "law_llm_descriptors_0000000_all.jsonl",
        )
        PATHS["court_v5"] = _first_existing(
            # local
            root / "artifacts" / "court_authority_cards_v5_unified.jsonl",
            # Drive (per _drive_pull: 'artifacts_v2/...')
            root / "artifacts_v2" / "court_authority_cards_v5_unified.jsonl",
        )
        PATHS["emb_dir"] = _first_existing(
            # local
            root / "embeddings",
            # Drive (per _drive_pull: 'artifacts/embeddings/')
            root / "artifacts" / "embeddings",
        )
        PATHS["emb_manifest"] = _first_existing(
            root / "embeddings" / "qwen3_8b_unified_manifest.parquet",
            root / "artifacts" / "embeddings" / "qwen3_8b_unified_manifest.parquet",
        )
        PATHS["graph_db"] = _first_existing(
            # local layout
            root / "data_insights" / "citation_graph_db_and_edges" / "citation_graph_extracted.sqlite",
            # Drive flat layout
            root / "data_insights" / "citation_graph_extracted.sqlite",
            # optional fallback
            root / "artifacts" / "citation_graph_extracted.sqlite",
        )

        # Diagnostic print so the caller knows which candidates resolved.
        print(f"[run_funnel] artifacts_root={root}")
        for k, p in PATHS.items():
            print(f"  {k:14s} -> {p}  {'OK' if p.exists() else 'MISSING'}")

    if config:
        CONFIG.update(config)

    # Map user's preferred keys to notebook keys (additive — original keys win).
    targets = dict(targets or {})
    key_alias = {
        "statutes":     "statute_targets",
        "concepts_en":  "concept_targets_en",
        "term_orig":    "term_targets_de",   # default DE; FR/IT can be passed explicitly
        "legal_areas":  "legal_area_keywords",
    }
    for src, dst in key_alias.items():
        if src in targets and dst not in targets:
            targets[dst] = targets[src]
    for k in ("statute_targets", "case_targets", "concept_targets_en",
              "term_targets_de", "term_targets_fr", "legal_area_keywords"):
        targets.setdefault(k, [])

    st = _get_state()
    _load_graph()
    _load_vector()  # silent no-op if torch / embeddings unavailable

    # --- Build canonical sets / code-family expansion -----------------------
    llm_statute_canons: set[str] = set()
    for raw in targets.get("statute_targets", []) or []:
        c = statute_anchor_canonical(raw)
        if c:
            llm_statute_canons.add(c)
    co_expanded_canons = set(llm_statute_canons)
    for canon in llm_statute_canons:
        for nb, _ in st["co_neighbours"].get(canon, []):
            co_expanded_canons.add(nb)

    statute_target_codes: set[str] = set()
    for canon in llm_statute_canons:
        if " " in canon:
            statute_target_codes.add(canon.split()[1])
    llm_codes_only = set(statute_target_codes)
    _kfam = CONFIG.get("code_family_top_k", 8)
    for c in llm_codes_only:
        related = sorted(
            ((cc, n) for (a, cc), n in st["code_pair_count"].items() if a == c),
            key=lambda x: -x[1])[:_kfam]
        for cc, _ in related:
            statute_target_codes.add(cc)

    # --- Concept expansion --------------------------------------------------
    llm_concepts = (list(targets.get("concept_targets_en") or [])
                    + list(targets.get("legal_area_keywords") or []))
    corpus_concept_keys = set(st["idx_concept_en"].keys())
    expanded_weighted = expand_concepts_weighted(
        llm_concepts, corpus_concept_keys, top_k=25,
    )

    # --- Topical channels --------------------------------------------------
    ch_law_direct = channel_law_direct(
        co_expanded_canons, st["idx_law_direct"], CONFIG["budget_law_direct"],
    )
    ch_court_stat = channel_court_statute(
        llm_statute_canons, st["idx_court_statute"],
        st["idx_court_statute_count"], st["doc_meta"],
        CONFIG["budget_court_statute"],
    )
    ch_concept = channel_concept(
        expanded_weighted, st["idx_concept_en"], CONFIG["budget_concept"],
    )
    ch_term = channel_term(
        targets, st["idx_term_orig"], st["idx_term_lemma"],
        CONFIG["budget_term"], corpus_keys=st["term_orig_keys"],
    )
    ch_per_area = channel_per_area_bedrock(
        targets.get("legal_area_keywords", []),
        statute_target_codes, st["per_area_canon_count"],
        st["idx_law_direct"], CONFIG["budget_per_area"],
    )
    ch_cocit = channel_co_citation(
        targets, st["co_neighbours"], st["idx_law_direct"],
        st["idx_court_statute"], st["canon_count"], CONFIG["budget_co_citation"],
    )

    # --- BM25 (multi-language; lang param biases which lang-specific targets
    # are emphasised, but the multi-lang merger still runs all 4 indices). ---
    ch_bm25 = bm25_search_multilang(query, targets, CONFIG["budget_bm25"])

    # --- Vector channels (raw + enriched). The lang parameter selects which
    # enrichment terms are appended; vector_raw always uses the query as-is. -
    if _VECTOR["ok"]:
        q_emb_raw = _encode_query(query)
        if lang == "de":
            enr_bits = (list(targets.get("term_targets_de") or [])
                        + list(targets.get("concept_targets_en") or []))
        elif lang == "fr":
            enr_bits = (list(targets.get("term_targets_fr") or [])
                        + list(targets.get("concept_targets_en") or []))
        elif lang == "it":
            enr_bits = (list(targets.get("term_targets_de") or [])
                        + list(targets.get("term_targets_fr") or [])
                        + list(targets.get("concept_targets_en") or []))
        else:  # "en"
            enr_bits = (list(targets.get("term_targets_de") or [])
                        + list(targets.get("term_targets_fr") or [])
                        + list(targets.get("concept_targets_en") or []))
        enriched_query = query + " " + " ".join(enr_bits[:60])
        q_emb_enr = _encode_query(enriched_query)
        ch_vector = _vector_search(q_emb_raw, CONFIG["budget_vector"])
        ch_venrich = _vector_search(q_emb_enr, CONFIG["budget_vector_enriched"])
    else:
        ch_vector, ch_venrich = [], []

    # --- Seed pool for graph + sibling + backprop --------------------------
    doc_meta = st["doc_meta"]
    def _court_hits(hits):
        return {d for d, _ in hits if doc_meta.get(d, {}).get("family") == "court"}
    seed = (
        _court_hits(ch_court_stat) | _court_hits(ch_law_direct)
        | _court_hits(ch_concept)  | _court_hits(ch_term)
        | _court_hits(ch_per_area) | _court_hits(ch_cocit)
        | _court_hits(ch_bm25)
        | _court_hits(ch_vector)   | _court_hits(ch_venrich)
    )

    ch_sibling = channel_sibling(
        seed, st["idx_court_base"], st["idx_judgment_importance"],
        st["doc_meta"], CONFIG["budget_sibling"],
    )

    if _GRAPH_LOADED and st["idx_graph_out"]:
        ch_graph_fwd = channel_graph_forward(
            seed, st["idx_graph_out"], st["idx_judgment_importance"],
            st["doc_meta"], CONFIG["budget_graph_forward"],
        )
        ch_graph_rev = channel_graph_reverse(
            seed, st["idx_graph_in"], st["idx_judgment_importance"],
            st["doc_meta"], CONFIG["budget_graph_reverse"],
        )
        if CONFIG["enable_graph_2hop"]:
            ch_graph_2h = channel_graph_2hop(
                seed, st["idx_graph_out"], CONFIG["budget_graph_2hop"],
            )
        else:
            ch_graph_2h = []
    else:
        ch_graph_fwd, ch_graph_rev, ch_graph_2h = [], [], []

    backprop_seed = (seed | _court_hits(ch_sibling)
                     | _court_hits(ch_graph_fwd) | _court_hits(ch_graph_rev))
    ch_backprop = channel_statute_backprop(
        backprop_seed, st["doc_statute_anchors"], st["idx_law_direct"],
        st["idx_court_statute_count"], CONFIG["budget_backprop"],
    )

    CHANNELS = [
        ("law_direct_match",  ch_law_direct),
        ("court_statute",     ch_court_stat),
        ("co_citation",       ch_cocit),
        ("per_area_bedrock",  ch_per_area),
        ("statute_backprop",  ch_backprop),
        ("sibling_expansion", ch_sibling),
        ("graph_forward",     ch_graph_fwd),
        ("graph_reverse",     ch_graph_rev),
        ("graph_2hop",        ch_graph_2h),
        ("concept_en",        ch_concept),
        ("term_orig",         ch_term),
        ("bm25",              ch_bm25),
        ("vector_raw",        ch_vector),
        ("vector_enriched",   ch_venrich),
    ]

    # --- Fusion -------------------------------------------------------------
    rrf_scores = rrf_fuse(
        CHANNELS, CONFIG["rrf_k"], weights=CONFIG.get("channel_weights"),
    )
    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    ranked_dids = [d for d, _ in ranked]
    ranked_gated = apply_neg_gate(ranked_dids, doc_meta, CONFIG["noise_paragraph_roles"])

    ch_by_name = dict(CHANNELS)
    guarantee = round_robin_guarantee(
        ch_by_name, CONFIG["guarantee_channels"],
        CONFIG.get("guarantee_per_channel", 130),
        top_k,
    )
    guarantee = apply_neg_gate(guarantee, doc_meta, CONFIG["noise_paragraph_roles"])
    guarantee_set = set(guarantee)

    final_topk = list(guarantee)
    seen_f = set(final_topk)
    for did in ranked_gated:
        if len(final_topk) >= top_k:
            break
        if did not in seen_f:
            final_topk.append(did); seen_f.add(did)
    final_topk = final_topk[:top_k]

    # --- DataFrame ----------------------------------------------------------
    # Per-channel rank/score lookups.
    channel_lookup: dict[str, dict[str, tuple[int, float]]] = {}
    for name, hits in CHANNELS:
        m: dict[str, tuple[int, float]] = {}
        for r, (d, s) in enumerate(hits):
            if d not in m:
                m[d] = (r, float(s) if isinstance(s, (int, float)) else 0.0)
        channel_lookup[name] = m

    channel_names = [n for n, _ in CHANNELS]

    rows: list[dict] = []
    for rank_idx, did in enumerate(final_topk):
        meta = doc_meta.get(did, {})
        row: dict = {
            "did":           did,
            "citation":      meta.get("citation", did),
            "family":        meta.get("family", ""),
            "rrf_rank":      rank_idx,
            "rrf_score":     float(rrf_scores.get(did, 0.0)),
            "in_guarantee":  did in guarantee_set,
        }
        ch_scores: dict[str, float | None] = {}
        for name in channel_names:
            pair = channel_lookup[name].get(did)
            if pair is None:
                row[f"rank_{name}"] = None
                row[f"score_{name}"] = None
                ch_scores[name] = None
            else:
                r, s = pair
                row[f"rank_{name}"] = r
                row[f"score_{name}"] = s
                ch_scores[name] = s
        row["channel_scores"] = ch_scores
        rows.append(row)

    df = pd.DataFrame(rows)
    return df


# =============================================================================
# 12. Smoke test — val_004 with a hand-built target dict.
# =============================================================================

def _smoke_test() -> None:
    """Load val_004 from val.csv, build a synthetic target dict (no LLM call),
    run the funnel, print top-20.
    """
    print("=" * 78)
    print("RUN_FUNNEL SMOKE TEST  --  val_004")
    print("=" * 78)
    val_df = pd.read_csv(PATHS["val_csv"])
    row = val_df[val_df["query_id"] == "val_004"].iloc[0]
    query = str(row["query"])
    gold = [c.strip() for c in str(row["gold_citations"]).split(";") if c.strip()]
    print(f"val_004: {len(gold)} gold citations")
    print(f"Query (first 200 chars): {query[:200]}{'...' if len(query) > 200 else ''}")

    # Synthetic targets — fake but realistic for val_004 (Swiss inheritance /
    # holographic will under ZGB / OR). The funnel needs SOMETHING in each
    # bucket; quality of LLM-generated targets only affects recall.
    targets = {
        "statute_targets": [
            "Art. 505 ZGB", "Art. 467 ZGB", "Art. 469 ZGB",
            "Art. 471 ZGB", "Art. 20 OR", "Art. 100 BGG",
        ],
        "case_targets": [],
        "concept_targets_en": [
            "holographic will", "testamentary capacity", "form requirements",
            "succession", "testator intent",
        ],
        "term_targets_de": [
            "eigenhändige Verfügung", "Testierfähigkeit", "Verfügung von Todes wegen",
            "Erbe", "Vermächtnis",
        ],
        "term_targets_fr": [
            "testament olographe", "capacité de tester", "disposition pour cause de mort",
        ],
        "legal_area_keywords": ["civil law", "inheritance", "succession"],
    }

    t = time.time()
    df = run_funnel(query, targets, lang="en", top_k=1000)
    print(f"\n[smoke] run_funnel returned {len(df)} rows in {time.time()-t:.1f}s")

    # Gold-in-pool diagnostic
    st = _get_state()
    gold_dids: set[str] = set()
    for g in gold:
        for d in st["cit_to_doc_ids"].get(g, []):
            gold_dids.add(d)
    in_pool = set(df["did"].tolist()) & gold_dids
    print(f"[smoke] gold mapped to doc_ids: {len(gold_dids)} / {len(gold)}")
    print(f"[smoke] gold in top-{len(df)}: {len(in_pool)} ({100*len(in_pool)/max(1,len(gold_dids)):.1f}%)")
    print(f"\nTop-20:")
    print(df.head(20)[["rrf_rank", "did", "citation", "family",
                       "rrf_score", "in_guarantee"]].to_string(index=False))


In [ ]:
# Phase 3b — sanity-check the inlined module
print(f"run_funnel defined:        {callable(run_funnel)}")
print(f"DEFAULT_ARTIFACTS_ROOT:    {DEFAULT_ARTIFACTS_ROOT}")
print(f"channel weights resolved:  {len(CONFIG['channel_weights'])} channels")
for ch in CONFIG["channel_weights"]:
    disabled = (ch == "graph_2hop" and not CONFIG.get("enable_graph_2hop", False))
    print(f"  - {ch:20s} weight={CONFIG['channel_weights'][ch]:.1f}  {'(disabled)' if disabled else ''}")


In [ ]:
# Phase 3c — run funnel per sub-issue, both languages, union to top-200

if cand_df is None:
    TOP_K_PER_LANG = 1000
    TOP_K_AFTER_UNION = 200

    all_rows = []
    for s in bundle["sub_issues"]:
        idx = s["idx"]
        targets = s["targets"]
        per_lang_results = []

        for lang in ("en", "de"):
            qtext = s["issue_en"] if lang == "en" else s["issue_de"]
            print(f"  [sub {idx}, {lang}] running funnel...")
            df_lang = run_funnel(
                query=qtext,
                targets=targets,
                lang=lang,
                top_k=TOP_K_PER_LANG,
                artifacts_root=FUNNEL_ART_ROOT,
            )
            df_lang["sub_issue_idx"] = idx
            df_lang["sub_issue_query_lang"] = lang
            per_lang_results.append(df_lang)

        # union both languages — keep best (lowest) rrf_rank per (did, sub_issue_idx)
        union_df = pd.concat(per_lang_results, ignore_index=True)
        union_df = (
            union_df.sort_values("rrf_rank")
                    .drop_duplicates(subset=["did"], keep="first")
                    .head(TOP_K_AFTER_UNION)
        )
        union_df["sub_issue_idx"] = idx
        all_rows.append(union_df)
        print(f"    -> {len(union_df)} candidates after union (top {TOP_K_AFTER_UNION})")

    cand_df = pd.concat(all_rows, ignore_index=True)
    cand_df.to_parquet(CKPT_CANDIDATES, index=False)
    print(f"\nSaved candidates -> {CKPT_CANDIDATES}  rows={len(cand_df):,}")


In [ ]:
# Phase 3d — recall check: of the parent_gold, how many are in the union across all sub-issue top-200s?
gold_set = set(PARENT_GOLD)
retrieved_cits = set(cand_df["citation"].dropna().astype(str))
caught = gold_set & retrieved_cits
print(f"\n=== Phase 3 recall check ===")
print(f"  parent gold:                          {len(gold_set)}")
print(f"  retrieved (union over all sub-pools): {len(retrieved_cits):,}")
print(f"  gold caught in any sub-pool:          {len(caught)} / {len(gold_set)} = {len(caught)/len(gold_set):.1%}")
print(f"  gold MISSED:                          {sorted(gold_set - caught)}")


## Phase 4 — Reranker scoring per sub-issue

Load Qwen3-Reranker-8B base + val_009 LoRA adapter, score the top-200 candidates per sub-issue, keep top-K for K ∈ {5, 8, 10}.

Reranker prompt structure (per `reranker_finetune_poc_val009_clean.ipynb`):
```
<|im_start|>system
Judge whether the Document meets the requirements based on the Query and the Instruct provided.
Note that the answer can only be "yes" or "no".
<|im_end|>
<|im_start|>user
<Instruct>: {instruction}
<Query>: {sub_issue_query} | target_codes: ... | target_areas: ...
<Document>: citation: ... | code: ... | area: ... | role: ... | title: ... | text: ...
<|im_end|>
<|im_start|>assistant
<think>

</think>
```

Score = softmax over (no_id, yes_id) at last position, take P(yes).

In [ ]:
# Phase 4a — skip check
if CKPT_RERANKED.exists():
    print(f"[skip] {CKPT_RERANKED} exists. Delete to re-run Phase 4.")
    rerank_df = pd.read_parquet(CKPT_RERANKED)
    print(f"  loaded {len(rerank_df):,} reranker scores")
else:
    rerank_df = None


In [ ]:
# Phase 4b — load base + LoRA
if rerank_df is None:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel

    BASE = "Qwen/Qwen3-Reranker-8B"
    print(f"Loading base: {BASE}")
    tok = AutoTokenizer.from_pretrained(BASE, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(
        BASE,
        torch_dtype=torch.bfloat16,
        device_map="cuda",
        attn_implementation="sdpa",
    )
    print(f"Loading LoRA adapter from {LORA_DIR}")
    model = PeftModel.from_pretrained(model, str(LORA_DIR))
    model.eval()

    # yes / no token ids
    yes_id = tok.encode("yes", add_special_tokens=False)[0]
    no_id  = tok.encode("no",  add_special_tokens=False)[0]
    print(f"yes_id={yes_id}, no_id={no_id}")


In [ ]:
# Phase 4c — build prompt from candidate row
INSTRUCTION = "Given a Swiss legal question, judge whether the Document is a relevant statute article or court decision that helps answer the question."

PROMPT_TEMPLATE = """<|im_start|>system
Judge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>
<|im_start|>user
<Instruct>: {instruction}
<Query>: {query_text} | target_codes: {codes} | target_areas: {areas} | target_concepts: {concepts}
<Document>: citation: {cit} | code: {code} | area: {area} | role: {role}
title: {title} | topic: {topic} | concepts: {doc_concepts}
text: {text}<|im_end|>
<|im_start|>assistant
<think>

</think>
"""

def build_prompt(sub_issue, cand_row):
    t = sub_issue["targets"]
    return PROMPT_TEMPLATE.format(
        instruction=INSTRUCTION,
        query_text=sub_issue["issue_en"],
        codes=", ".join(t.get("legal_areas", [])),
        areas=", ".join(t.get("legal_areas", [])),
        concepts=", ".join(t.get("concepts_en", [])[:8]),
        cit=cand_row.get("citation", ""),
        code=cand_row.get("law_code", "") or cand_row.get("court_base", ""),
        area=cand_row.get("legal_area", ""),
        role=cand_row.get("role", ""),
        title=str(cand_row.get("law_title", ""))[:150],
        topic=cand_row.get("topic", ""),
        doc_concepts=cand_row.get("concepts", ""),
        text=str(cand_row.get("text", ""))[:1000],
    )


In [ ]:
# Phase 4d — score in batches, per sub-issue
if rerank_df is None:
    from tqdm.auto import tqdm
    MAX_LEN = 1536
    BATCH = 8

    @torch.no_grad()
    def score_batch(prompts):
        enc = tok(prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to("cuda")
        out = model(**enc, logits_to_keep=1)
        last = out.logits[:, -1, :]
        yes_l = last[:, yes_id].float()
        no_l  = last[:, no_id].float()
        probs = torch.nn.functional.softmax(torch.stack([no_l, yes_l], dim=1), dim=1)[:, 1]
        return probs.cpu().tolist()

    scored_rows = []
    for s in bundle["sub_issues"]:
        idx = s["idx"]
        sub_df = cand_df[cand_df.sub_issue_idx == idx].reset_index(drop=True)
        print(f"  sub {idx}: scoring {len(sub_df)} candidates")
        prompts = [build_prompt(s, sub_df.iloc[i]) for i in range(len(sub_df))]
        scores = []
        for i in tqdm(range(0, len(prompts), BATCH), leave=False):
            scores.extend(score_batch(prompts[i:i+BATCH]))
        sub_df["rerank_score"] = scores
        scored_rows.append(sub_df)

    rerank_df = pd.concat(scored_rows, ignore_index=True)
    rerank_df.to_parquet(CKPT_RERANKED, index=False)
    print(f"\nSaved reranker scores -> {CKPT_RERANKED}  rows={len(rerank_df):,}")


## Phase 5 — Aggregate → Kaggle exact-string F1

For each K_per_issue ∈ {5, 8, 10}:
1. Per sub-issue, keep top-K by `rerank_score`
2. Union across sub-issues
3. Canonicalize each citation per the Kaggle grader (`.strip()` + whitespace collapse)
4. Compare to canonicalized gold → F1

This matches the official `scripts/evaluate_submission.py` from the Omnilex starter repo exactly.

In [ ]:
# Phase 5a — canonicalizer (exactly matches Kaggle grader)
import re as _re
_WS_RE = _re.compile(r"\s+")
def canon(c):
    return _WS_RE.sub(" ", c.strip())

def f1_set(pred, gold):
    p, g = set(pred), set(gold)
    if not p and not g: return 1.0
    if not p or not g:  return 0.0
    tp = len(p & g)
    if not tp: return 0.0
    pr = tp / len(p); rc = tp / len(g)
    return 2 * pr * rc / (pr + rc)

gold_canon = {canon(c) for c in PARENT_GOLD}
print(f"Gold size (canonicalized): {len(gold_canon)}")


In [ ]:
# Phase 5b — try K_per_issue ∈ {5, 8, 10} and report
results = {}
for K in (5, 8, 10):
    per_issue_top = (
        rerank_df.sort_values("rerank_score", ascending=False)
                 .groupby("sub_issue_idx")
                 .head(K)
    )
    pred = {canon(c) for c in per_issue_top["citation"].dropna().astype(str)}
    f1 = f1_set(pred, gold_canon)
    tp = len(pred & gold_canon)
    fp = len(pred - gold_canon)
    fn = len(gold_canon - pred)
    pr = tp / max(tp + fp, 1)
    rc = tp / max(tp + fn, 1)
    results[K] = dict(K=K, n_pred=len(pred), tp=tp, fp=fp, fn=fn, precision=pr, recall=rc, f1=f1)
    print(f"  K_per_issue={K:2d}  |pred|={len(pred):3d}  TP={tp:2d}  FP={fp:3d}  FN={fn:2d}  P={pr:.3f}  R={rc:.3f}  F1={f1:.4f}")

# best K
best = max(results.values(), key=lambda r: r["f1"])
print(f"\n>>> Best: K_per_issue={best['K']}, F1={best['f1']:.4f} (precision={best['precision']:.3f}, recall={best['recall']:.3f})")


In [ ]:
# Phase 5c — diagnostics: which gold caught, which missed (at best K)
import json as _json
best_K = best["K"]
per_issue_top = (
    rerank_df.sort_values("rerank_score", ascending=False)
             .groupby("sub_issue_idx")
             .head(best_K)
)
pred_canon = {canon(c) for c in per_issue_top["citation"].dropna().astype(str)}

caught_gold = sorted(gold_canon & pred_canon)
missed_gold = sorted(gold_canon - pred_canon)
false_pos   = sorted(pred_canon - gold_canon)

print(f"=== Diagnostics at K_per_issue={best_K} ===")
print(f"\n[caught] {len(caught_gold)}/{len(gold_canon)} gold")
for c in caught_gold:
    print(f"  {c}")
print(f"\n[missed] {len(missed_gold)}")
for c in missed_gold:
    in_pool = c in {canon(x) for x in cand_df.citation.dropna().astype(str)}
    where = "[in funnel pool]" if in_pool else "[NOT in funnel pool — Phase 3 failure]"
    print(f"  {c}  {where}")
print(f"\n[false positives] {len(false_pos)} (showing first 20)")
for c in false_pos[:20]:
    print(f"  {c}")


In [ ]:
# Phase 5d — save F1 result + per-sub-issue contributions
contrib = []
for idx in sorted(rerank_df.sub_issue_idx.unique()):
    sub_top = (
        rerank_df[rerank_df.sub_issue_idx == idx]
            .sort_values("rerank_score", ascending=False)
            .head(best_K)
    )
    sub_pred = {canon(c) for c in sub_top.citation.dropna().astype(str)}
    sub_caught = sorted(gold_canon & sub_pred)
    contrib.append({
        "sub_issue_idx": int(idx),
        "issue_en": bundle["sub_issues"][int(idx)]["issue_en"],
        "n_pred": len(sub_pred),
        "n_gold_caught": len(sub_caught),
        "gold_caught": sub_caught,
    })

summary = {
    "qid": QID,
    "parent_query_len": len(PARENT_QUERY),
    "n_gold": len(PARENT_GOLD),
    "n_sub_issues": len(bundle["sub_issues"]),
    "results_by_K": results,
    "best_K": best["K"],
    "best_f1": best["f1"],
    "per_sub_issue_contribution": contrib,
}
CKPT_F1.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved -> {CKPT_F1}")
print(f"\nBest F1 for {QID}: {best['f1']:.4f}  (K_per_issue={best['K']})")


## Appendix — what to inspect after the run

1. `{QID}_bundle.json` — did the LLM decompose into a reasonable number of issues? Are issues atomic? Do targets look right?
2. `{QID}_candidates.parquet` — Phase 3 recall check: how many gold cits are in the union of sub-issue funnel pools? If << 0.85, the funnel parameterization is undercatching — investigate before scaling.
3. `{QID}_reranked.parquet` — top-K per sub-issue. Look at `rerank_score` distribution: bimodal (sharp yes/no) is good; flat is bad.
4. `{QID}_f1.json` — per-sub-issue contribution. If any sub-issue contributes 0 gold catches, that issue's decomposition or targets need work.

## Next steps after PoC

- **If F1 ≥ 0.4** on val_004: scale to all 10 val queries. Same notebook, change `QID` and re-run.
- **If 0.2 ≤ F1 < 0.4**: the architecture works but reranker needs help. Options:
  - Fine-tune a fresh LoRA on `train.csv` (1139 single-issue queries, no val leakage)
  - Add a verification pass (LLM "does this article answer this issue?" yes/no)
- **If F1 < 0.2**: decomposition or funnel is the bottleneck.
  - Inspect Phase 3 recall: if union of pools doesn't contain gold, fix the funnel parameterization
  - If pools contain gold but reranker doesn't surface it, fix reranker prompt / use base model without LoRA

## val_009 leakage note

This PoC uses val_004 specifically because the val_009 LoRA was trained on val_009 candidates. When we scale to all 10 val queries, report:
- Macro-F1 across all 10 (optimistic — includes val_009 leakage)
- Macro-F1 across 9 (exclude val_009 — honest)
- Per-query F1 so val_009's contribution is visible
